<div style="padding:1.15rem 1.25rem;border:1px solid #273246;border-radius:18px;background:linear-gradient(135deg,#111827 0%,#172033 58%,#0f3a33 100%);color:#f8fafc;margin-bottom:0.9rem;">
  <div style="display:flex;gap:0.45rem;flex-wrap:wrap;margin-bottom:0.75rem;">
    <span style="padding:0.22rem 0.6rem;border-radius:999px;background:rgba(71,180,117,0.16);border:1px solid rgba(71,180,117,0.42);font-size:0.76rem;font-weight:600;">Dataset Prep</span>
    <span style="padding:0.22rem 0.6rem;border-radius:999px;background:rgba(7,173,248,0.12);border:1px solid rgba(7,173,248,0.34);font-size:0.76rem;font-weight:600;">Viewer Refresh</span>
    <span style="padding:0.22rem 0.6rem;border-radius:999px;background:rgba(255,180,0,0.14);border:1px solid rgba(255,180,0,0.34);font-size:0.76rem;font-weight:600;">Training Handoff</span>
  </div>
  <h1 style="margin:0 0 0.35rem 0;color:#ffffff;">v7 DSL Dataset Preparation</h1>
  <p style="margin:0;max-width:56rem;line-height:1.58;">Prepare a split-aware SVG/DSL workspace, stage it into a run-local dataset snapshot, refresh the dataset viewer and IR hub, and surface the exact next-step training commands without leaving the notebook.</p>
</div>

## Notebook Flow

1. Inspect the seed workspace under `version/v7/data/spec04`.
2. Materialize workspace artifacts and stage them into `RUN/dataset/`.
3. Open `dataset_viewer.html`, `ir_hub.html`, and any run-local reports.
4. Hand the staged dataset into `TrainingProject.materialize()` or `ck_run_v7.py init`.

## Start Here

- Launch from the repo root with `.venv/bin/jupyter lab notebooks/v7_dsl_dataset_preparation.ipynb`.
- The command cells are safe by default; nothing heavy runs until you flip the `EXECUTE_*` flags.
- Keep the repo workspace as the seed template and the cache-backed run directory as the operator artifact home.


In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys
from IPython.display import HTML, display

REPO_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ckernel_engine").exists() and (candidate / "version" / "v7").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError(
        f"Could not find the C-Kernel-Engine repo root from cwd={Path.cwd()}. "
        "Open this notebook from inside the repo checkout."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PYTHON_EXEC = Path(sys.executable).resolve()
V7_ROOT = REPO_ROOT / "version" / "v7"
print(f"Repo root: {REPO_ROOT}")
print(f"Python exec: {PYTHON_EXEC}")


In [ ]:
def shell_join(command):
    return shlex.join(str(part) for part in command)


def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def run_and_print(command, *, check=True):
    rendered = [str(part) for part in command]
    print("$ " + shell_join(rendered))
    completed = subprocess.run(
        rendered,
        cwd=str(REPO_ROOT),
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout.rstrip())
    if completed.stderr:
        print(completed.stderr.rstrip())
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit code {completed.returncode}")
    return completed


def existing_uri(path):
    path = Path(path)
    return path.resolve().as_uri() if path.exists() else None


def artifact_board_html(*, title, subtitle, entries):
    cards = []
    for entry in entries:
        path = Path(entry["path"]) if entry.get("path") else None
        exists = bool(path and path.exists())
        accent = "#47b475" if exists else entry.get("accent", "#ffb400")
        status = "Ready" if exists else entry.get("missing_label", "Not generated yet")
        href = path.resolve().as_uri() if exists else None
        action_html = (
            f"<a href='{href}' style='color:#8cb4ff;text-decoration:none;font-weight:700;'>Open artifact</a>"
            if href
            else f"<span style='color:#8b97ab;'>{entry.get('hint', 'Create this in the previous step.')}</span>"
        )
        path_html = f"<code>{path}</code>" if path else "<span style='color:#8b97ab;'>Path decided at runtime</span>"
        cards.append(
            f"""
            <div style='border:1px solid #253146;border-radius:14px;background:#101722;padding:0.9rem 0.95rem;box-shadow:0 8px 24px rgba(0,0,0,0.18);'>
                <div style='display:flex;align-items:center;justify-content:space-between;gap:0.6rem;margin-bottom:0.45rem;'>
                    <strong style='color:#f8fafc;font-size:0.95rem;'>{entry['label']}</strong>
                    <span style='padding:0.18rem 0.5rem;border-radius:999px;background:{accent}22;border:1px solid {accent}66;color:{accent};font-size:0.72rem;font-weight:700;'>{status}</span>
                </div>
                <div style='font-size:0.78rem;color:#b6c2d1;line-height:1.45;margin-bottom:0.55rem;'>{entry.get('description', '')}</div>
                <div style='font-size:0.72rem;color:#8b97ab;line-height:1.45;margin-bottom:0.7rem;'>{path_html}</div>
                <div style='font-size:0.8rem;'>{action_html}</div>
            </div>
            """
        )
    return f"""
    <div style='margin:0.25rem 0 0.9rem;'>
        <div style='margin-bottom:0.75rem;'>
            <h3 style='margin:0 0 0.2rem 0;color:#0f172a;font-size:1.02rem;'>{title}</h3>
            <p style='margin:0;color:#516072;font-size:0.86rem;line-height:1.45;'>{subtitle}</p>
        </div>
        <div style='display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:0.7rem;'>
            {''.join(cards)}
        </div>
    </div>
    """


In [ ]:
WORKSPACE = REPO_ROOT / "version" / "v7" / "data" / "spec04"
RUN_DIR = Path.home() / ".cache" / "ck-engine-v7" / "models" / "train" / "v7-dsl-dataset-notebook-demo"
DATASET_ROOT = RUN_DIR / "dataset"
DATASET_VIEWER = RUN_DIR / "dataset_viewer.html"
DATASET_SNAPSHOT = DATASET_ROOT / "dataset_snapshot.json"
MODELS_ROOT = RUN_DIR.parent.parent

{
    "workspace": str(WORKSPACE),
    "run_dir": str(RUN_DIR),
    "dataset_root": str(DATASET_ROOT),
    "dataset_viewer": str(DATASET_VIEWER),
    "models_root": str(MODELS_ROOT),
}


## Inspect The Seed Workspace

The split-aware seed workspace lives under `version/v7/data/spec04`. It is the repo template, not the final operator artifact home.

<div style="padding:0.8rem 0.95rem;border-left:4px solid #47b475;background:rgba(71,180,117,0.08);border-radius:10px;margin:0.6rem 0 0.85rem 0;">
<strong>Operator model:</strong> the repo workspace is the editable seed, while the run directory is the durable artifact home.
</div>

Use this section to confirm:
- which manifests and split folders already exist
- what is still just source workspace structure
- what will later move into the run-local dataset snapshot


In [ ]:
workspace_entries = sorted(p.name for p in WORKSPACE.iterdir()) if WORKSPACE.exists() else []
manifest_dir = WORKSPACE / "manifests"
manifest_summaries = {}
if manifest_dir.exists():
    for path in sorted(manifest_dir.glob("*.json")):
        payload = read_json(path)
        if isinstance(payload, dict):
            manifest_summaries[path.name] = payload.get("schema") or payload.get("format") or sorted(payload.keys())[:5]
        else:
            manifest_summaries[path.name] = "unreadable"

{
    "workspace_exists": WORKSPACE.exists(),
    "workspace_entries": workspace_entries,
    "manifests": manifest_summaries,
}


## Materialize And Stage The Workspace

The notebook keeps the command surface explicit so the user can see the real v7 tools, not a hidden wrapper.

<div style="padding:0.85rem 0.95rem;border:1px solid #253146;border-radius:12px;background:#f7fafc;margin-bottom:0.75rem;">
<strong>Safe by default:</strong> the next cell builds the commands, and the execution cell after that will not run them until you set the `EXECUTE_*` flags to `True`.
</div>

The main workflow is:
1. one-time workspace scaffold: `init_data_workspace_v7.sh`
2. refresh workspace artifacts: `materialize_svg_stage_artifacts_v7.py`
3. stage the workspace into a run and build `dataset_viewer.html`: `stage_dataset_workspace_v7.py`
4. refresh the shared `ir_hub.html`: `open_ir_hub.py`

That keeps the notebook useful both as a launcher and as living documentation for the exact dataset path.


In [ ]:
INIT_WORKSPACE_CMD = [
    "bash",
    str(REPO_ROOT / "version" / "v7" / "scripts" / "init_data_workspace_v7.sh"),
    "--spec",
    "spec04",
    "--dataset-type",
    "svg",
]

MATERIALIZE_WORKSPACE_CMD = [
    str(PYTHON_EXEC),
    str(REPO_ROOT / "version" / "v7" / "scripts" / "materialize_svg_stage_artifacts_v7.py"),
    "--workspace",
    str(WORKSPACE),
    "--force",
]

STAGE_WORKSPACE_CMD = [
    str(PYTHON_EXEC),
    str(REPO_ROOT / "version" / "v7" / "scripts" / "dataset" / "stage_dataset_workspace_v7.py"),
    "--workspace",
    str(WORKSPACE),
    "--run-dir",
    str(RUN_DIR),
    "--mode",
    "copy",
    "--force",
]

REBUILD_VIEWER_CMD = [
    str(PYTHON_EXEC),
    str(REPO_ROOT / "version" / "v7" / "scripts" / "build_svg_dataset_visualizer_v7.py"),
    "--workspace",
    str(DATASET_ROOT),
    "--output",
    str(DATASET_VIEWER),
]

REFRESH_HUB_CMD = [
    str(PYTHON_EXEC),
    str(REPO_ROOT / "version" / "v7" / "tools" / "open_ir_hub.py"),
    "--models-root",
    str(MODELS_ROOT),
    "--output",
    str(MODELS_ROOT / "ir_hub.html"),
    "--index-out",
    str(MODELS_ROOT / "runs_hub_index.json"),
]

{
    "init_workspace": shell_join(INIT_WORKSPACE_CMD),
    "materialize_workspace": shell_join(MATERIALIZE_WORKSPACE_CMD),
    "stage_workspace": shell_join(STAGE_WORKSPACE_CMD),
    "rebuild_viewer": shell_join(REBUILD_VIEWER_CMD),
    "refresh_hub": shell_join(REFRESH_HUB_CMD),
}


In [ ]:
EXECUTE_MATERIALIZE_WORKSPACE = False
EXECUTE_STAGE_WORKSPACE = False
EXECUTE_REFRESH_HUB = False

if EXECUTE_MATERIALIZE_WORKSPACE:
    run_and_print(MATERIALIZE_WORKSPACE_CMD)

if EXECUTE_STAGE_WORKSPACE:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    run_and_print(STAGE_WORKSPACE_CMD)
    if not DATASET_VIEWER.exists():
        print("dataset_viewer.html was not created during staging; try REBUILD_VIEWER_CMD explicitly.")

if EXECUTE_REFRESH_HUB:
    run_and_print(REFRESH_HUB_CMD)

if not any([EXECUTE_MATERIALIZE_WORKSPACE, EXECUTE_STAGE_WORKSPACE, EXECUTE_REFRESH_HUB]):
    print("Set one or more EXECUTE_* flags to True to run the scaffolded dataset commands.")


## Inspect Staged Dataset Artifacts

The next cell renders a small artifact board instead of a plain link list.

After staging, the key files to inspect are:
- `RUN/dataset/dataset_snapshot.json`
- `RUN/dataset/manifests/*workspace_manifest.json`
- `RUN/dataset_viewer.html`
- `MODELS_ROOT/ir_hub.html`

If `ir_report.html` exists too, that means you already handed the run off into the training/materialization path.


In [ ]:
snapshot = read_json(DATASET_SNAPSHOT)
workspace_manifest_candidates = sorted((DATASET_ROOT / "manifests").glob("*workspace_manifest.json")) if DATASET_ROOT.exists() else []
workspace_manifest_path = workspace_manifest_candidates[0] if workspace_manifest_candidates else None
artifact_entries = [
    {
        "label": "Dataset Viewer",
        "path": DATASET_VIEWER,
        "description": "Run-local HTML viewer for the staged dataset workspace.",
        "hint": "Stage the workspace or run the viewer rebuild command first.",
    },
    {
        "label": "Dataset Snapshot",
        "path": DATASET_SNAPSHOT,
        "description": "JSON summary of what the staging step copied into the run.",
        "hint": "The stage step writes this snapshot into RUN/dataset/.",
        "accent": "#8cb4ff",
    },
    {
        "label": "Workspace Manifest",
        "path": workspace_manifest_path,
        "description": "Manifest view of the staged dataset layout and split metadata.",
        "hint": "Look for *workspace_manifest.json under RUN/dataset/manifests/.",
        "accent": "#8cb4ff",
    },
    {
        "label": "IR Report",
        "path": RUN_DIR / "ir_report.html",
        "description": "Available after the training/materialization handoff creates run-level IR artifacts.",
        "hint": "Materialize the run to generate the visualizer report.",
        "accent": "#ffb400",
    },
    {
        "label": "IR Hub",
        "path": MODELS_ROOT / "ir_hub.html",
        "description": "Shared dashboard across all cache-backed runs under the models root.",
        "hint": "Run the hub refresh command to regenerate the shared index.",
        "accent": "#47b475",
    },
]
display(
    HTML(
        artifact_board_html(
            title="Dataset Artifact Board",
            subtitle="Open the run-local dataset outputs and the shared run dashboards from one place.",
            entries=artifact_entries,
        )
    )
)

{
    "dataset_viewer_exists": DATASET_VIEWER.exists(),
    "dataset_snapshot_exists": DATASET_SNAPSHOT.exists(),
    "workspace_manifest": str(workspace_manifest_path) if workspace_manifest_path else None,
    "staged_entries": snapshot.get("staged_entries", []) if isinstance(snapshot, dict) else None,
    "missing_entries": snapshot.get("missing_entries", []) if isinstance(snapshot, dict) else None,
}


## Handoff Into Training

This notebook stops at dataset preparation on purpose, but it should still show the next hop into the real v7 training surface.

<div style="padding:0.8rem 0.95rem;border-left:4px solid #ffb400;background:rgba(255,180,0,0.08);border-radius:10px;margin:0.6rem 0 0.85rem 0;">
<strong>Next step:</strong> keep dataset prep separate from the full training operator flow, but surface the exact Python and CLI handoff commands here so users do not have to translate paths manually.
</div>

Two handoff styles matter:
- Python authoring path: `TrainingProject(...).materialize(MaterializeOptions(dataset_workspace=...))`
- CLI path: `ck_run_v7.py init --dataset-workspace ... --dataset-stage-force ...`

The cell below builds both without running a full train. It also surfaces the stage-specific train files that the workspace currently exposes.


In [ ]:
from ckernel_engine.v7 import MaterializeOptions, TemplateSpec, TinyModelSpec, TokenizerPlan, TrainingProject

train_data_candidates = {
    "pretrain_train": [str(path) for path in sorted((WORKSPACE / "pretrain" / "train").glob("*.txt"))[:5]],
    "midtrain_train": [str(path) for path in sorted((WORKSPACE / "midtrain" / "train").glob("*.txt"))[:5]],
    "sft_train": [str(path) for path in sorted((WORKSPACE / "sft" / "train").glob("*.txt"))[:5]],
}

project = TrainingProject(
    run_name=RUN_DIR.name,
    run_dir=RUN_DIR,
    model=TinyModelSpec(
        init="xavier_uniform",
        layers=2,
        vocab_size=256,
        embed_dim=128,
        hidden_dim=256,
        num_heads=8,
        num_kv_heads=4,
        context_len=128,
    ),
    template=TemplateSpec.builtin_template("qwen3"),
    tokenizer=TokenizerPlan(
        family="runtime_default",
        notes="Dataset notebook handoff example.",
    ),
)

materialize_options = MaterializeOptions(
    generate_ir=True,
    generate_runtime=True,
    strict=True,
    dataset_workspace=WORKSPACE,
    dataset_stage_mode="copy",
    dataset_stage_force=True,
)

cli_init_cmd = [
    str(PYTHON_EXEC),
    str(REPO_ROOT / "version" / "v7" / "scripts" / "ck_run_v7.py"),
    "init",
    "--run",
    str(RUN_DIR),
    "--run-name",
    RUN_DIR.name,
    "--template",
    "qwen3",
    "--generate-ir",
    "--generate-runtime",
    "--strict",
    "--dataset-workspace",
    str(WORKSPACE),
    "--dataset-stage-mode",
    "copy",
    "--dataset-stage-force",
    *project.model.to_init_args(),
]

{
    "python_materialize_options": materialize_options.to_metadata(),
    "cli_init_command": shell_join(cli_init_cmd),
    "pretrain_train_files": train_data_candidates["pretrain_train"],
    "midtrain_train_files": train_data_candidates["midtrain_train"],
    "sft_train_files": train_data_candidates["sft_train"],
}
